# Panel de viaje · Sesión Overpass · Pruebas pendientes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tragabytes/panel-viaje/blob/main/tests/fase1_overpass.ipynb)

**Proyecto:** https://github.com/tragabytes/panel-viaje

Sesión específica de Overpass. Cubre las tres tareas que la sesión 05 dejó pendientes por saturación de mirrors:

1. **Prueba 0** — salud de los mirrors. Si los tres están caídos otra vez, paramos limpio.
2. **Prueba 5 (bloque 1.4)** — pueblos cercanos a un punto GPS. Crítica: alimenta la vista 3.
3. **Prueba 3 (bloque 1.4)** — Overpass como fallback de POIs vs Wikidata.
4. **Próxima salida de autovía (bloque 1.2, segunda parte)** — `motorway_link` cercanos filtrados por heading.

Formato compacto. Cada prueba es independiente tras el setup.

## Setup

In [ ]:
import requests, time, math

UA = "panel-viaje-tragabytes/0.1 (https://github.com/tragabytes/panel-viaje; contacto via GitHub)"
HEADERS_OV = {"User-Agent": UA}

# Tres mirrors públicos conocidos. La prueba 0 elige el que responda.
MIRRORS = [
    ("overpass-api.de",       "https://overpass-api.de/api/interpreter"),
    ("kumi.systems",          "https://overpass.kumi.systems/api/interpreter"),
    ("private.coffee",        "https://overpass.private.coffee/api/interpreter"),
]

# Endpoint activo. La prueba 0 lo rellena.
OVERPASS_OK = None

def fetch_overpass(query, endpoint=None, max_intentos=4, timeout=60):
    """Reintentos espaciados; mismo patrón validado en sesión 05."""
    url = endpoint or OVERPASS_OK
    if not url:
        return {"ok": False, "error": "sin endpoint Overpass disponible", "intentos": 0}
    esperas = [0, 5, 15, 30]
    last_err = None
    for intento in range(max_intentos):
        if esperas[intento]: time.sleep(esperas[intento])
        t0 = time.time()
        try:
            r = requests.post(url, data={"data": query}, headers=HEADERS_OV, timeout=timeout)
            dt = (time.time() - t0) * 1000
            if r.status_code == 200:
                return {"ok": True, "data": r.json(), "ms": dt, "bytes": len(r.content), "intentos": intento + 1}
            last_err = f"HTTP {r.status_code}: {r.text[:120]}"
        except Exception as e:
            last_err = f"{type(e).__name__}: {str(e)[:120]}"
    return {"ok": False, "error": last_err, "intentos": max_intentos}

def distancia_m(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1); dl = math.radians(lon2 - lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def bearing_deg(lat1, lon1, lat2, lon2):
    """Rumbo en grados desde el punto 1 hacia el punto 2 (0=N, 90=E)."""
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dl = math.radians(lon2 - lon1)
    y = math.sin(dl) * math.cos(p2)
    x = math.cos(p1)*math.sin(p2) - math.sin(p1)*math.cos(p2)*math.cos(dl)
    return (math.degrees(math.atan2(y, x)) + 360) % 360

def diff_angular(a, b):
    """Diferencia angular mínima entre dos rumbos en grados (0..180)."""
    d = abs(a - b) % 360
    return d if d <= 180 else 360 - d

print("Setup OK")

## Prueba 0 — Salud de los mirrors

Query mínima a cada uno. El primero que responda con 200 será `OVERPASS_OK` para el resto del notebook. Si los tres fallan, las celdas siguientes lo detectan y abortan.

In [ ]:
QUERY_PING = '[out:json][timeout:10];out count;'

resultados_ping = []
for nombre, url in MIRRORS:
    t0 = time.time()
    try:
        r = requests.post(url, data={"data": QUERY_PING}, headers=HEADERS_OV, timeout=15)
        dt = (time.time() - t0) * 1000
        ok = r.status_code == 200
        msg = f"HTTP {r.status_code}" if not ok else f"OK · {dt:.0f} ms"
    except Exception as e:
        ok = False
        msg = f"{type(e).__name__}: {str(e)[:60]}"
    resultados_ping.append((nombre, url, ok, msg))
    estado = "✓" if ok else "✗"
    print(f"  {estado} {nombre:20s}  {msg}")

vivos = [(n, u) for n, u, ok, _ in resultados_ping if ok]
if vivos:
    OVERPASS_OK = vivos[0][1]
    print(f"\nMirror elegido: {vivos[0][0]}  ({len(vivos)}/3 disponibles)")
else:
    print("\n⚠ Los tres mirrors están caídos. Las celdas siguientes abortarán limpiamente.")
    print("  Acción: reintentar la sesión más tarde.")

## Prueba 5 — Pueblos cercanos a un punto GPS

La más crítica: sin esto no hay vista 3. Radio 15 km, `place=village|town|city`, top 4 por distancia.

3 puntos en carretera real (no sobre pueblo) para simular el caso del coche circulando.

In [ ]:
PUNTOS_CARRETERA = [
    {"nombre": "A-2 cerca de Medinaceli", "lat": 41.1856, "lon": -2.4570},
    {"nombre": "A-5 cerca de Trujillo",   "lat": 39.4760, "lon": -5.8200},
    {"nombre": "N-VI cerca de Astorga",   "lat": 42.4420, "lon": -6.0800},
]

def query_pueblos_cercanos(lat, lon, radio_m=15000):
    return f'''[out:json][timeout:50];
(
  node["place"~"^(village|town|city)$"](around:{radio_m},{lat},{lon});
);
out tags;'''

resultados_cercanos = {}

if not OVERPASS_OK:
    print("⚠ Sin mirror activo. Saltando prueba 5.")
else:
    for p in PUNTOS_CARRETERA:
        print(f"\n=== {p['nombre']} ({p['lat']}, {p['lon']}) ===")
        res = fetch_overpass(query_pueblos_cercanos(p['lat'], p['lon'], 15000))
        if not res['ok']:
            print(f"  ERROR: {res['error'][:120]}")
            resultados_cercanos[p['nombre']] = {"ok": False, "error": res['error']}
            continue
        elementos = res['data'].get('elements', [])
        pueblos = []
        for e in elementos:
            tags = e.get('tags', {})
            name = tags.get('name')
            if not name: continue
            d = distancia_m(p['lat'], p['lon'], e.get('lat', 0), e.get('lon', 0))
            pueblos.append({"name": name, "place": tags.get('place'), "dist": d, "pop": tags.get('population')})
        pueblos.sort(key=lambda x: x['dist'])
        sin_nombre = sum(1 for el in elementos if not el.get('tags', {}).get('name'))
        print(f"  {len(pueblos)} con nombre · {sin_nombre} sin nombre · {res['ms']:.0f} ms · {res['bytes']} B · {res['intentos']} intento(s)")
        print("  Top 10 por distancia:")
        for pu in pueblos[:10]:
            pop = f"pop={pu['pop']}" if pu['pop'] else ""
            print(f"    {pu['dist']/1000:5.1f} km  {pu['name'][:32]:32s} [{pu['place']}] {pop}")
        resultados_cercanos[p['nombre']] = {
            "ok": True, "ms": res['ms'], "bytes": res['bytes'],
            "total": len(pueblos), "top4": pueblos[:4],
        }
        time.sleep(3)  # cortesía con Overpass

## Prueba 3 — Overpass como fallback de POIs vs Wikidata

Mismos 5 municipios de la sesión 05. Radio 2 km. Tags: `historic=*`, `tourism=viewpoint|attraction`, `natural=peak`, `building=castle|church|monastery|cathedral|chapel`.

Referencia Wikidata sesión 05 (POIs / con imagen): Medinaceli 7/7, Albarracín 28/19, Chinchón 12/12, Trujillo 21/14, Pedraza 3/3. La pregunta es si Overpass aporta algo en Pedraza (caso límite).

In [ ]:
MUNICIPIOS_POI = [
    {"nombre": "Medinaceli", "lat": 41.1731, "lon": -2.4344, "wd_pois": 7,  "wd_img": 7},
    {"nombre": "Albarracín", "lat": 40.4064, "lon": -1.4433, "wd_pois": 28, "wd_img": 19},
    {"nombre": "Chinchón",   "lat": 40.1396, "lon": -3.4261, "wd_pois": 12, "wd_img": 12},
    {"nombre": "Trujillo",   "lat": 39.4600, "lon": -5.8825, "wd_pois": 21, "wd_img": 14},
    {"nombre": "Pedraza",    "lat": 41.1283, "lon": -3.8108, "wd_pois": 3,  "wd_img": 3},
]

def query_pois_cercanos(lat, lon, radio_m=2000):
    return f'''[out:json][timeout:50];
(
  node["historic"](around:{radio_m},{lat},{lon});
  way["historic"](around:{radio_m},{lat},{lon});
  node["tourism"="viewpoint"](around:{radio_m},{lat},{lon});
  node["tourism"="attraction"](around:{radio_m},{lat},{lon});
  way["tourism"="attraction"](around:{radio_m},{lat},{lon});
  node["natural"="peak"](around:{radio_m},{lat},{lon});
  way["building"~"^(castle|church|monastery|cathedral|chapel)$"](around:{radio_m},{lat},{lon});
);
out center tags;'''

resultados_ov_pois = {}

if not OVERPASS_OK:
    print("⚠ Sin mirror activo. Saltando prueba 3.")
else:
    for m in MUNICIPIOS_POI:
        print(f"\n=== {m['nombre']} (Wikidata sesión 05: {m['wd_pois']} POIs, {m['wd_img']} con imagen) ===")
        res = fetch_overpass(query_pois_cercanos(m['lat'], m['lon'], 2000))
        if not res['ok']:
            print(f"  ERROR: {res['error'][:120]}")
            resultados_ov_pois[m['nombre']] = {"ok": False, "error": res['error']}
            continue
        elementos = res['data'].get('elements', [])
        con_nombre = [e for e in elementos if e.get('tags', {}).get('name')]
        print(f"  {len(elementos)} elem · {len(con_nombre)} con name · {res['ms']:.0f} ms · {res['bytes']} B · {res['intentos']} intento(s)")
        for e in con_nombre[:15]:
            tags = e.get('tags', {})
            name = tags.get('name', '?')
            tipo = ' '.join(f"{k}={tags[k]}" for k in ('historic','tourism','natural','building') if k in tags)
            print(f"    {name[:42]:42s}  [{tipo}]")
        if len(con_nombre) > 15:
            print(f"    ... y {len(con_nombre)-15} más")
        resultados_ov_pois[m['nombre']] = {
            "ok": True, "ms": res['ms'], "bytes": res['bytes'],
            "total": len(elementos), "con_nombre": len(con_nombre), "elementos": con_nombre,
        }
        time.sleep(3)

print("\n--- Comparativa Wikidata vs Overpass ---")
print(f"{'Municipio':14s} {'WD POIs':>9s} {'WD img':>8s} {'OV elem':>9s} {'OV nom':>8s}")
for m in MUNICIPIOS_POI:
    r = resultados_ov_pois.get(m['nombre'], {})
    if r.get('ok'):
        print(f"{m['nombre']:14s} {m['wd_pois']:>9d} {m['wd_img']:>8d} {r['total']:>9d} {r['con_nombre']:>8d}")
    else:
        print(f"{m['nombre']:14s} {m['wd_pois']:>9d} {m['wd_img']:>8d}    ERROR")

## Próxima salida de autovía (bloque 1.2, segunda parte)

**Estrategia:** dado un punto GPS + heading del coche, buscar `highway=motorway_link` (los enlaces de salida son siempre motorway_link en OSM) en un radio de 5 km, filtrar los que están **delante** del coche según el heading, ordenar por distancia, devolver el más cercano.

**Filtrado geométrico:** para cada enlace, calcular el rumbo desde el punto del coche hasta el enlace y comparar con el heading. Si la diferencia angular es menor que 90°, está delante. Tolerancia configurable.

**Headings inventados** coherentes con la dirección normal de la vía:
- A-2 Medinaceli: ~60° (NE, dirección Zaragoza/Barcelona).
- A-5 Trujillo: ~250° (WSW, dirección Mérida/Badajoz).
- N-VI Astorga: ~290° (WNW, dirección Galicia).

Los nodos de `motorway_link` no traen `name` casi nunca; los que sí información llevan son los `way` con `destination`, `destination:ref`, `ref` y `oneway`. Pedimos `way` con `out center tags` para tener un punto representativo y los tags.

In [ ]:
PUNTOS_AUTOVIA = [
    {"nombre": "A-2 cerca de Medinaceli", "lat": 41.1856, "lon": -2.4570, "heading": 60,  "sentido": "NE → Zaragoza"},
    {"nombre": "A-5 cerca de Trujillo",   "lat": 39.4760, "lon": -5.8200, "heading": 250, "sentido": "WSW → Mérida"},
    {"nombre": "N-VI cerca de Astorga",   "lat": 42.4420, "lon": -6.0800, "heading": 290, "sentido": "WNW → Galicia"},
]

def query_motorway_links(lat, lon, radio_m=5000):
    return f'''[out:json][timeout:50];
(
  way["highway"="motorway_link"](around:{radio_m},{lat},{lon});
);
out center tags;'''

def analizar_salidas(punto, elementos, tolerancia_deg=90):
    """Para cada enlace devuelto, calcula distancia y si está delante."""
    candidatos = []
    for e in elementos:
        center = e.get('center') or {}
        elat, elon = center.get('lat'), center.get('lon')
        if elat is None or elon is None: continue
        d = distancia_m(punto['lat'], punto['lon'], elat, elon)
        rumbo_a = bearing_deg(punto['lat'], punto['lon'], elat, elon)
        diff = diff_angular(rumbo_a, punto['heading'])
        delante = diff < tolerancia_deg
        tags = e.get('tags', {})
        candidatos.append({
            "id": e.get('id'),
            "dist": d,
            "rumbo_a": rumbo_a,
            "diff_heading": diff,
            "delante": delante,
            "ref":         tags.get('ref'),
            "destination": tags.get('destination'),
            "dest_ref":    tags.get('destination:ref'),
            "dest_street": tags.get('destination:street'),
            "junction_ref":tags.get('junction:ref'),
        })
    return sorted(candidatos, key=lambda x: x['dist'])

def agrupar_por_salida(candidatos_delante):
    """Varios way pueden pertenecer al mismo nudo de salida. Agrupa por destination/ref/junction_ref normalizado."""
    grupos = {}
    for c in candidatos_delante:
        clave = c.get('junction_ref') or c.get('dest_ref') or c.get('destination') or c.get('ref') or f"sin_clave_{c['id']}"
        if clave not in grupos or c['dist'] < grupos[clave]['dist']:
            grupos[clave] = c
    return sorted(grupos.values(), key=lambda x: x['dist'])

resultados_salidas = {}

if not OVERPASS_OK:
    print("⚠ Sin mirror activo. Saltando prueba de próxima salida.")
else:
    for p in PUNTOS_AUTOVIA:
        print(f"\n=== {p['nombre']}  heading={p['heading']}° ({p['sentido']}) ===")
        res = fetch_overpass(query_motorway_links(p['lat'], p['lon'], 5000))
        if not res['ok']:
            print(f"  ERROR: {res['error'][:120]}")
            resultados_salidas[p['nombre']] = {"ok": False, "error": res['error']}
            continue
        elementos = res['data'].get('elements', [])
        candidatos = analizar_salidas(p, elementos)
        delante = [c for c in candidatos if c['delante']]
        agrupados = agrupar_por_salida(delante)
        print(f"  {len(elementos)} motorway_link en 5 km · {len(delante)} delante · {len(agrupados)} salidas únicas · {res['ms']:.0f} ms · {res['bytes']} B")
        if not agrupados:
            print("  ⚠ ningún motorway_link delante. Posibles causas: heading invertido, punto fuera de autovía, o salidas más allá del radio.")
            print("  Top 5 candidatos sin filtrar (todos los rumbos):")
            for c in candidatos[:5]:
                marca = "→" if c['delante'] else " "
                etiqueta = c['junction_ref'] or c['dest_ref'] or c['destination'] or c['ref'] or '(sin tags útiles)'
                print(f"    {marca} {c['dist']/1000:5.2f} km  rumbo={c['rumbo_a']:5.1f}°  Δ={c['diff_heading']:5.1f}°  {etiqueta[:50]}")
        else:
            print("  Próximas salidas (agrupadas, ordenadas por distancia):")
            for c in agrupados[:5]:
                etiqueta = c['junction_ref'] or c['dest_ref'] or c['destination'] or c['ref'] or '(sin tags útiles)'
                print(f"    {c['dist']/1000:5.2f} km  Δheading={c['diff_heading']:5.1f}°  {etiqueta[:60]}")
        resultados_salidas[p['nombre']] = {
            "ok": True, "ms": res['ms'], "bytes": res['bytes'],
            "total_links": len(elementos), "delante": len(delante),
            "salidas_unicas": len(agrupados), "top": agrupados[:3],
        }
        time.sleep(3)

## Resumen final

Copia este bloque al chat para que Claude actualice el documento de seguimiento.

In [ ]:
print("="*70)
print("RESUMEN SESIÓN OVERPASS")
print("="*70)

print("\n## P0 — Salud de mirrors")
for nombre, url, ok, msg in resultados_ping:
    print(f"  {'✓' if ok else '✗'} {nombre:20s}  {msg}")
print(f"  Mirror elegido: {OVERPASS_OK or 'NINGUNO'}")

print("\n## P5 — Pueblos cercanos a punto GPS (radio 15 km)")
if not resultados_cercanos:
    print("  (no ejecutada)")
for nombre, r in resultados_cercanos.items():
    if r.get('ok'):
        print(f"  {nombre:32s} {r['total']:3d} pueblos · {r['ms']:.0f} ms · {r['bytes']} B")
        for pu in r['top4']:
            print(f"    {pu['dist']/1000:5.1f} km  {pu['name']}")
    else:
        print(f"  {nombre:32s} ERROR")

print("\n## P3 — Overpass como fallback de POIs (radio 2 km)")
if not resultados_ov_pois:
    print("  (no ejecutada)")
else:
    print(f"  {'Municipio':14s} {'WD POIs':>9s} {'WD img':>8s} {'OV elem':>9s} {'OV nom':>8s}")
    for m in MUNICIPIOS_POI:
        r = resultados_ov_pois.get(m['nombre'], {})
        if r.get('ok'):
            print(f"  {m['nombre']:14s} {m['wd_pois']:>9d} {m['wd_img']:>8d} {r['total']:>9d} {r['con_nombre']:>8d}")
        else:
            print(f"  {m['nombre']:14s} {m['wd_pois']:>9d} {m['wd_img']:>8d}    ERROR")

print("\n## Próxima salida de autovía (radio 5 km, motorway_link, filtrado por heading)")
if not resultados_salidas:
    print("  (no ejecutada)")
for nombre, r in resultados_salidas.items():
    if r.get('ok'):
        print(f"  {nombre:32s} {r['total_links']:3d} links · {r['delante']:3d} delante · {r['salidas_unicas']:3d} salidas únicas · {r['ms']:.0f} ms")
        for c in r['top']:
            etiqueta = c.get('junction_ref') or c.get('dest_ref') or c.get('destination') or c.get('ref') or '(sin tags)'
            print(f"    {c['dist']/1000:5.2f} km  {etiqueta[:50]}")
    else:
        print(f"  {nombre:32s} ERROR")

print("\n" + "="*70)
print("Pega este resumen en el chat para análisis y actualización del seguimiento.")